# Compact preprocessing: validated trees to the same canonical groups

Compact output drops repeated step rows while retaining exactly the quantities required by the
historical analysis. It cannot recover arbitrary per-step trajectories. This notebook imports
production schema validation and reconstruction; it implements no second grouping algorithm.
The default example is explicitly synthetic and executable without a campaign.

## Tables and keys

- `Groups`: one historical group, keyed by `(EventNumber, GroupIndex)`. Final particle/nucleus/process
  labels, admitted StepCount, distinct TrackCount and VolumeCount accompany it.
- `GroupVolumes`: one row per group/gas copy, with consecutive zero-based VolumeOrder. Energy is the
  ordered step sum in MeV. First x/y/z and FirstHitTime_ns come from the **same first admitted step**.
- `Tracks`: one event-local track entering sensitive gas, with first-sensitive-step diagnostic labels.
  GroupIndex is -1 for ignored particles, a nonnegative single group, or -2 for multiple groups.
- `TrackGroups`: explicit `(EventNumber, ParticleID, GroupIndex)` membership for every admitted track.
  Historical groups may merge multiple tracks; a resumed track can belong to multiple groups.

FirstHitTime_ns is event-relative pre-step Geant4 time, not an absolute clock shared by primaries.
It is retained for future intra-event coincidence studies and inactive in canonical analysis.


In [1]:
from pathlib import Path
import sys, inspect, tempfile, json, copy
import numpy as np
# Start Jupyter in the repository or any subdirectory (also works in archived source/).
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "study/analyze.py").is_file())
if str(REPO) not in sys.path: sys.path.insert(0, str(REPO))
try:
    from IPython.display import display, Markdown, Image
except ImportError:  # lightweight cell execution test; Jupyter itself supplies these
    display = print
    Markdown = str
from analysis.common.examples import IDENTITY, GEOMETRY
import uproot
from analysis.common.examples import write_example
from analysis.common.io import output_metadata
# Optional real worker file: supply its matching identity/accounting/geometry too.
INPUT_FILE = None
expected = IDENTITY
requested = 2
geometry = GEOMETRY
example_directory = tempfile.TemporaryDirectory(prefix="cygno-notebook-")
if INPUT_FILE is None:
    INPUT_FILE = Path(example_directory.name) / "synthetic-both.root"
    write_example(INPUT_FILE)
    print("SYNTHETIC SOFTWARE EXAMPLE — not a Monte Carlo rate estimate")
with uproot.open(INPUT_FILE) as file:
    print(output_metadata(file))


SYNTHETIC SOFTWARE EXAMPLE — not a Monte Carlo rate estimate
{'OutputFormat': 'both', 'OutputSchemaVersion': 1, 'TimingDefinition': 'pre-step global time / ns; Geant4 event-relative, not inter-event time', 'CompactProcessingVersion': 'event-boundaries-and-eof-v2'}


## Inspect schemas, counts and representative records

In [2]:
from analysis.compact.io import header, SCHEMAS
from analysis.compact.preprocess import groups, reconstruct
from analysis.common.model import Group
with uproot.open(INPUT_FILE) as file:
    accounting, trees = header(file, expected, requested)
    for name, tree in trees.items():
        print(name, SCHEMAS[name], "rows:", tree.num_entries)
        display(tree.arrays(entry_stop=5, library="np"))


Groups {'EventNumber': 'int32', 'GroupIndex': 'int32', 'ParticleName': 'string', 'Nucleus': 'string', 'ProcessType': 'string', 'StepCount': 'int32', 'TrackCount': 'int32', 'VolumeCount': 'int32'} rows: 2
{'EventNumber': array([0, 1], dtype=int32), 'GroupIndex': array([0, 0], dtype=int32), 'ParticleName': array(['e-', 'e-'], dtype=object), 'Nucleus': array(['synthetic nucleus', 'synthetic nucleus'], dtype=object), 'ProcessType': array(['RadioactiveDecay', 'RadioactiveDecay'], dtype=object), 'StepCount': array([3, 1], dtype=int32), 'TrackCount': array([2, 1], dtype=int32), 'VolumeCount': array([2, 1], dtype=int32)}
GroupVolumes {'EventNumber': 'int32', 'GroupIndex': 'int32', 'VolumeOrder': 'int32', 'VolumeNumber': 'int32', 'EnergyDeposit': 'float64', 'x_first': 'float64', 'y_first': 'float64', 'z_first': 'float64', 'FirstHitTime_ns': 'float64'} rows: 3
{'EventNumber': array([0, 0, 1], dtype=int32), 'GroupIndex': array([0, 0, 0], dtype=int32), 'VolumeOrder': array([0, 1, 0], dtype=int32),

## Reconstruction and validation

The reader merges four ordered tree streams one event at a time. It rejects duplicate keys,
missing groups or tracks, invalid volume order, nonfinite values, invalid event/track identities,
inconsistent sentinels, and count mismatches. Parent tracks need not appear in sensitive gas.
No metadata field is guessed from the number of branches. An unsupported output schema or
processing version is rejected. Scientific identity is checked separately.


In [3]:
print(inspect.getsource(reconstruct))
with uproot.open(INPUT_FILE) as file:
    _, trees = header(file, expected, requested)
    tracks = []
    canonical = list(groups(trees, requested, expected['layout'], step_size=2, track_sink=tracks.append))
for group in canonical[:5]:
    assert isinstance(group, Group)
    display(group)


def reconstruct(event, data, layout):
    groups = {}
    for row in data['Groups']:
        key = row['GroupIndex']
        require(key == len(groups), 'Duplicate/out-of-order GroupIndex')
        require(row['ParticleName'] in ('e-','e+','alpha'), 'Invalid group particle')
        require(row['StepCount'] >= row['TrackCount'] > 0 and
                row['StepCount'] >= row['VolumeCount'] > 0, 'Invalid group counts')
        groups[key] = Group(event, row['ParticleName'], row['Nucleus'], row['ProcessType'],
                            group_index=key, step_count=row['StepCount'])
    previous = (-1,-1)
    for row in data['GroupVolumes']:
        key = row['GroupIndex']
        require(key in groups, 'Orphan group volume')
        group = groups[key]
        volume = row['VolumeNumber']
        order = row['VolumeOrder']
        require((key,order) > previous and order==len(group.volumes), 'Invalid VolumeOrder')
        require(volume not in group.volumes, 'Duplicate group/volume key'

## Optional same-transport raw/compact parity

A `both` file is the definitive comparison: its raw and compact trees consumed the same StepRecords.
The production validator demands exact group/volume values, labels and ordering, track relationships,
first times, group energies, fiducial decisions, exact windows and all bins/flows. No energy tolerance
is introduced. Separate same-seed transport tests check that enabling compact output consumes no RNG.


In [4]:
from analysis.compact.parity import validate
with uproot.open(INPUT_FILE) as file:
    if output_metadata(file)['OutputFormat'] == 'both':
        display(validate(file, expected, requested, geometry, step_size=2))
    else:
        print("Provide a both file to execute same-transport parity.")


{'status': 'passed', 'comparison': 'exact', 'processing_version': 'event-boundaries-and-eof-v2', 'groups': 2, 'tracks': 4, 'group_volumes': 3, 'checks': ['group identities/order/labels/counts', 'volume order/energy/first position/first time', 'track provenance/membership', 'group energy/fiducial/window flags', 'all 900 bins/underflow/overflow/exact counts']}


## Handoff to common analysis

The following call is identical to the raw notebook's call. Provenance and timing do not enter it.
Compact production requires no separate raw ROOT file. For new scientific selections, retain a
separate version and document which historical assumptions were intentionally changed.


In [5]:
from analysis.common.spectra import accumulate
counts, histogram, n = accumulate(canonical, geometry)
assert np.all(counts[:, 0] == histogram.sum(axis=1))
print("Canonical groups:", n, "ER groups:", counts[0, 0])


Canonical groups: 2 ER groups: 2
